In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import pandas as pd
import sympy
import sympy.printing

In [2]:
# Read data
read_data_005=pd.read_csv("D:/Man/Earth and Environment/Manchester_data003-008/005_manchester_processed.csv")
read_data_005=read_data_005.sort_values(by='time').reset_index(drop=True)

learning_data_005=read_data_005[(read_data_005['year']>=2006)&(read_data_005['year']<=2050)].copy()
learning_data_005=learning_data_005.drop(columns=['lat','lon','date'],errors='ignore')

learning_data_005

,time,TREFMXAV_U,FLNS,FSNS,PRECT,PRSN,QBOT,TREFHT,UBOT,VBOT,year,month,day,dayofyear
0,2006-01-02,11.15260,8.737288,7.855509,1.544200e-07,4.051599e-16,0.005373,6.07195,2.303055,4.502803,2006,1,2,2
1,2006-01-03,13.01327,6.686464,7.501073,7.784098e-08,0.000000e+00,0.007595,11.04843,4.657475,3.158464,2006,1,3,3
2,2006-01-04,13.16030,27.445148,12.188718,4.851411e-08,7.075068e-18,0.005667,9.27023,5.083677,5.835154,2006,1,4,4
3,2006-01-05,14.85882,10.443632,4.354691,1.091676e-07,1.429017e-14,0.007362,11.82208,3.029053,8.031938,2006,1,5,5
4,2006-01-06,10.78576,70.927300,31.532597,2.531008e-09,6.418103e-18,0.004160,7.14828,3.783906,6.986506,2006,1,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15811,2050-12-27,10.58290,19.547508,15.106448,2.213155e-08,6.438513e-15,0.005779,8.01800,0.623059,4.167736,2050,12,361,361
15812,2050-12-28,14.16284,18.568079,13.015922,3.699803e-08,9.786361e-23,0.006798,9.50298,-1.815245,4.800050,2050,12,362,362
15813,2050-12-29,16.11770,15.843533,8.881753,2.557600e-08,1.473843e-22,0.008161,13.01970,-2.800902,7.744351,2050,12,363,363
15814,2050-12-30,13.72466,21.278680,9.996034,4.276372e-08,1.328554e-12,0.006569,10.77773,-1.253943,6.456124,2050,12,364,364


In [3]:
from sklearn.preprocessing import StandardScaler

features = ['FLNS', 'FSNS', 'PRECT', 'PRSN', 'QBOT', 'TREFHT', 'UBOT', 'VBOT']


X_005 = learning_data_005[features].copy()
y_005 = learning_data_005['TREFMXAV_U'].values

X_005['month_sin'] = np.sin(2 * np.pi * learning_data_005['month'] / 12)
X_005['month_cos'] = np.cos(2 * np.pi * learning_data_005['month'] / 12)

X_005['day_sin'] = np.sin(2 * np.pi * learning_data_005['dayofyear'] / 365.25)
X_005['day_cos'] = np.cos(2 * np.pi * learning_data_005['dayofyear'] / 365.25)

scaler_X = StandardScaler()
X_scaled_005 = scaler_X.fit_transform(X_005.values)

train_mask = learning_data_005['year'] <= 2040

X_train_raw = X_scaled_005[train_mask]
y_train_raw = y_005[train_mask]

X_val_raw = X_scaled_005[~train_mask]
y_val_raw = y_005[~train_mask]

print(f"Training set: {len(X_train_raw)}")
print(f"Validation set: {len(X_val_raw)}")

Training set: 12311
Validation set: 3505


In [4]:
# Time-series Spliting
def create_sequences(X, y, window_size=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i : i + window_size])
        y_seq.append(y[i + window_size])
    return torch.FloatTensor(np.array(X_seq)).transpose(1, 2), torch.FloatTensor(np.array(y_seq)).view(-1, 1)

In [5]:
# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0):
        self.patience = patience 
        self.verbose = verbose
        self.counter = 0   
        self.best_score = None 
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        self.best_state_dict = model.state_dict()
        self.val_loss_min = val_loss

# 1D-CNN
One convolutional layer with 32 out channels

window_size=30

learning rate=0.001

epoch=200

patience=15

In [7]:
class TemperatureCNN(nn.Module):
    def __init__(self, input_channels, window_size):
        super(TemperatureCNN, self).__init__()
        
        self.conv_layer = nn.Sequential(
            nn.Conv1d(in_channels=input_channels, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2) if window_size > 2 else nn.Identity(),
        )
        
        final_len = window_size // 2 if window_size > 2 else window_size
        self.flatten_dim = 32 * final_len
        
        self.fc_layer = nn.Sequential(
            nn.Linear(self.flatten_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [8]:
import torch.nn.functional as F

window_size = 30
X_train_t, y_train_t = create_sequences(X_train_raw, y_train_raw, window_size)
X_val_t, y_val_t = create_sequences(X_val_raw, y_val_raw, window_size)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_channels = X_train_t.shape[1]
model = TemperatureCNN(input_channels, window_size).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=15, factor=0.5)
criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)


for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    scheduler.step(val_mae)
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break


model.load_state_dict(early_stopping.best_state_dict)
model.eval()
with torch.no_grad():
    final_preds = model(X_val_t)
    mae = criterion(final_preds, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds, y_val_t).item())

print("\n" + "="*30)
print(f"CNN Final Result (2042-2050):")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print("="*30)

EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
Epoch 10: Train Loss 2.0367, Val MAE 1.7363
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 20: Train Loss 1.9329, Val MAE 1.6864
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
Epoch 30: Train Loss 1.8708, Val MAE 1.5786
EarlyStopping counter: 6 out of 15
EarlyStopping counter: 7 out of 15
EarlyStopping counter: 8 out of 15
EarlyStopping counter: 9 out of 15
EarlyStopping counter: 10 ou

# Transformer
One encoder layer

d_model=32

window size=30

learning rate=0.001

epoch=200

patience=15

In [9]:
class TemperatureTransformer(nn.Module):
    def __init__(self, input_channels, window_size, d_model=32, nhead=4, num_layers=1, dropout=0.1):
        super(TemperatureTransformer, self).__init__()
        
        self.input_fc = nn.Linear(input_channels, d_model)
        
        self.pos_encoder = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, 
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_fc = nn.Sequential(
            nn.Linear(d_model * window_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x shape: [Batch, Channels, Window] -> [Batch, Window, Channels]
        x = x.transpose(1, 2)
        x = self.input_fc(x) + self.pos_encoder
        x = self.transformer_encoder(x)
        x = x.reshape(x.size(0), -1) 
        return self.output_fc(x)

In [ ]:
# Training

window_size = 30

all_mae = []
all_rmse = []

X_train_t, y_train_t = create_sequences(X_train_raw, y_train_raw, window_size)
X_val_t, y_val_t = create_sequences(X_val_raw, y_val_raw, window_size)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_channels = X_train_t.shape[1]
model = TemperatureTransformer(input_channels, window_size).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)


for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    scheduler.step(val_mae)
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break


model.load_state_dict(early_stopping.best_state_dict)
model.eval()
with torch.no_grad():
    final_preds = model(X_val_t)
    mae = criterion(final_preds, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds, y_val_t).item())

print("\n" + "="*30)
print(f"Transformer Final Result (2042-2050):")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print("="*30)

EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 10: Train Loss 2.0204, Val MAE 1.6359
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
Epoch 20: Train Loss 1.9299, Val MAE 1.5565
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 5 out of 15
EarlyStopping counter: 6 out of 15
Epoch 30: Train Loss 1.8732, Val MAE 1.5229
EarlyStopping counter: 7 out of 15
EarlyStopping counter: 8 out of 15
EarlyStopping counter: 9 out of 15
EarlyStopping counter: 10 out of 15
EarlyStopping counter: 11 o

# LSTM
hidden_size=32

window size=30

learning rate=0.001

epoch=200

patience=15

In [10]:
class TemperatureLSTM(nn.Module):
    def __init__(self, input_channels, hidden_size=32, num_layers=1, dropout=0.2):
        super(TemperatureLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_channels, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2) 
        
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        
        out = out[:, -1, :]
        
        return self.fc(out)

In [11]:
window_size = 30
X_train_t, y_train_t = create_sequences(X_train_raw, y_train_raw, window_size)
X_val_t, y_val_t = create_sequences(X_val_raw, y_val_raw, window_size)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_channels = X_train_t.shape[1]

model = TemperatureLSTM(input_channels, hidden_size=32, num_layers=2).to(device)


optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

criterion = nn.L1Loss()
early_stopping = EarlyStopping(patience=15, verbose=True) 

X_val_t, y_val_t = X_val_t.to(device), y_val_t.to(device)


for epoch in range(200):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # 标准四步走：
        optimizer.zero_grad()           
        outputs = model(batch_X)        
        loss = criterion(outputs, batch_y)
        loss.backward()                 
        optimizer.step()              
        
        total_loss += loss.item()
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        val_mae = criterion(val_preds, y_val_t).item()
    
    early_stopping(val_mae, model)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss {total_loss/len(train_loader):.4f}, Val MAE {val_mae:.4f}")
    
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break


model.load_state_dict(early_stopping.best_state_dict)
model.eval()
with torch.no_grad():
    final_preds = model(X_val_t)
    mae = criterion(final_preds, y_val_t).item()
    rmse = np.sqrt(F.mse_loss(final_preds, y_val_t).item())

print("\n" + "="*30)
print(f"LSTM Final Result (2041-2050):")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print("="*30)

EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
Epoch 10: Train Loss 2.1268, Val MAE 1.8597
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
Epoch 20: Train Loss 1.9380, Val MAE 1.7118
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 4 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
Epoch 30: Train Loss 1.8202, Val MAE 1.6033
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
EarlyStopping counter: 1 out of 15
EarlyStopping counter: 2 out of 15
EarlyStopping counter: 3 out of 15
Epoch 40: Train Loss 1.7221, Val MAE 1.5741
EarlyStopping count